# Part 4: Pruning 종합 실험 및 분석

## 이 노트북에서 다루는 내용
1. 모든 Pruning 방법 통합 실험
2. Iterative Pruning (점진적 프루닝)
3. Unstructured vs Structured 비교
4. 종합 시각화 및 분석

---

## 실험 구성
1. **Baseline**: Pruning 없이 학습된 모델
2. **L1 Unstructured (30%)**: 개별 가중치 magnitude 기반 제거
3. **Global Unstructured (30%)**: 전역 magnitude 기반 제거
4. **L1 Structured (30%)**: 필터 단위 L1 norm 기반 제거
5. **L1 Structured + Fine-tune**: Pruning 후 재학습
6. **Iterative Pruning**: 점진적 pruning + fine-tuning 반복

## 1. 환경 설정 및 Import

In [ ]:
import os
import copy

import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

from module.models import CNN

os.makedirs('results', exist_ok=True)

In [ ]:
def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = get_device()
print(f"Using device: {DEVICE}")

BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 0.001
NUM_CLASSES = 10
SEED = 42

FINETUNE_EPOCHS = 3
FINETUNE_LR = 0.0001

## 2. 데이터 로드 및 유틸리티 함수

In [ ]:
def get_data_loaders(batch_size=128):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    train_dataset = datasets.CIFAR10(root='./data/datasets', train=True, download=True, transform=transform)
    test_dataset = datasets.CIFAR10(root='./data/datasets', train=False, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=True if DEVICE == "cuda" else False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=0, pin_memory=True if DEVICE == "cuda" else False)
    return train_loader, test_loader


def train_baseline(model, train_loader, epochs, lr, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.to(device)
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"  Epoch {epoch + 1}/{epochs}, Loss: {running_loss / len(train_loader):.4f}")


def test(model, test_loader, device):
    model.to(device)
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, predicted = torch.max(model(inputs).data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total


def fine_tune(model, train_loader, epochs, lr, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.to(device)
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"  Fine-tune {epoch + 1}/{epochs}, Loss: {running_loss / len(train_loader):.4f}")


def remove_pruning(model):
    for _, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            try:
                prune.remove(module, 'weight')
            except ValueError:
                pass
    return model


def get_sparsity(model):
    total = zeros = 0
    for _, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w = module.weight.data
            total += w.numel()
            zeros += (w == 0).sum().item()
    return zeros / total * 100 if total > 0 else 0

In [ ]:
print("[1] 데이터 로드...")
train_loader, test_loader = get_data_loaders(BATCH_SIZE)

## 3. Baseline 모델 준비

In [ ]:
print("[2] Baseline 모델 준비...")
model = CNN(num_classes=NUM_CLASSES).to(DEVICE)

try:
    model.load_state_dict(torch.load('./data/trained_models/baseline_model.pth', map_location=DEVICE, weights_only=True))
    print("    저장된 Baseline 로드 완료")
except FileNotFoundError:
    print("    Baseline 학습 중...")
    torch.manual_seed(SEED)
    train_baseline(model, train_loader, EPOCHS, LEARNING_RATE, DEVICE)
    torch.save(model.state_dict(), './data/trained_models/baseline_model.pth')

all_accuracies = {}
baseline_acc = test(model, test_loader, DEVICE)
all_accuracies['Baseline'] = baseline_acc
print(f"Baseline Accuracy: {baseline_acc:.2f}%")

## 4. 모든 Pruning 방법 적용 (30%)

In [ ]:
# L1 Unstructured (30%)
print("\n[3] L1 Unstructured (30%)...")
m = copy.deepcopy(model)
for _, module in m.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        prune.l1_unstructured(module, name='weight', amount=0.3)
m = remove_pruning(m)
acc = test(m, test_loader, DEVICE)
all_accuracies['L1 Unstructured (30%)'] = acc
print(f"Accuracy: {acc:.2f}%")

In [ ]:
# Global Unstructured (30%)
print("\n[4] Global Unstructured (30%)...")
m = copy.deepcopy(model)
params = [(module, 'weight') for _, module in m.named_modules() if isinstance(module, (nn.Conv2d, nn.Linear))]
prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=0.3)
m = remove_pruning(m)
acc = test(m, test_loader, DEVICE)
all_accuracies['Global Unstructured (30%)'] = acc
print(f"Accuracy: {acc:.2f}%")

In [ ]:
# L1 Structured (30%)
print("\n[5] L1 Structured (30%)...")
m = copy.deepcopy(model)
for _, module in m.named_modules():
    if isinstance(module, nn.Conv2d):
        prune.ln_structured(module, name='weight', amount=0.3, n=1, dim=0)
m = remove_pruning(m)
acc = test(m, test_loader, DEVICE)
all_accuracies['L1 Structured (30%)'] = acc
print(f"Accuracy: {acc:.2f}%")

In [ ]:
# L1 Structured (30%) + Fine-tune
print("\n[6] L1 Structured (30%) + Fine-tune...")
m = copy.deepcopy(model)
for _, module in m.named_modules():
    if isinstance(module, nn.Conv2d):
        prune.ln_structured(module, name='weight', amount=0.3, n=1, dim=0)
# mask 유지한 채 fine-tuning (pruned 가중치 0 유지)
fine_tune(m, train_loader, FINETUNE_EPOCHS, FINETUNE_LR, DEVICE)
# fine-tuning 완료 후 mask 영구 적용
m = remove_pruning(m)
acc = test(m, test_loader, DEVICE)
all_accuracies['L1 Structured + Fine-tune'] = acc
print(f"Accuracy: {acc:.2f}%")

## 5. Iterative Pruning (점진적 프루닝)

한 번에 많이 pruning하면 정확도가 크게 하락하므로,
**조금씩 pruning하고 fine-tuning하는 과정을 반복**합니다.

```
목표: 50% sparsity, 5회 반복
→ 매 회 ~13% pruning + fine-tuning
→ (1 - 0.87)^5 ≈ 0.50
```

In [ ]:
def iterative_pruning(model, train_loader, test_loader, target_sparsity, num_iterations, ft_epochs, ft_lr, device):
    """Iterative Pruning: 점진적으로 pruning + fine-tuning 반복"""
    per_iter_amount = 1 - (1 - target_sparsity) ** (1 / num_iterations)
    history = []
    
    print(f"    목표 Sparsity: {target_sparsity * 100:.1f}%")
    print(f"    반복 횟수: {num_iterations}")
    print(f"    매 반복 pruning 비율: {per_iter_amount * 100:.1f}%")

    for i in range(num_iterations):
        print(f"\n    --- Iteration {i + 1}/{num_iterations} ---")
        
        # Pruning (mask 유지)
        for _, module in model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                prune.l1_unstructured(module, name='weight', amount=per_iter_amount)

        # Pruning 후 정확도 (mask 유지 상태에서 평가)
        prune_acc = test(model, test_loader, device)
        sparsity = get_sparsity(model)
        print(f"    Pruning 후: Accuracy {prune_acc:.2f}%, Sparsity {sparsity:.1f}%")

        # Fine-tuning (mask 유지 → pruned 가중치 0 유지)
        fine_tune(model, train_loader, ft_epochs, ft_lr, device)
        ft_acc = test(model, test_loader, device)
        print(f"    Fine-tune 후: Accuracy {ft_acc:.2f}%")

        # Fine-tuning 완료 후 mask 영구 적용 (다음 iteration을 위해)
        model = remove_pruning(model)

        history.append({
            'iteration': i + 1, 'sparsity': sparsity,
            'prune_accuracy': prune_acc, 'finetune_accuracy': ft_acc
        })

    return model, history

In [ ]:
print("\n[7] Iterative Pruning (50% target, 5 iterations)...")
iter_model = copy.deepcopy(model)
iter_model, iter_history = iterative_pruning(
    iter_model, train_loader, test_loader,
    target_sparsity=0.5, num_iterations=5,
    ft_epochs=FINETUNE_EPOCHS, ft_lr=FINETUNE_LR, device=DEVICE
)
iter_acc = test(iter_model, test_loader, DEVICE)
all_accuracies['Iterative (50% target)'] = iter_acc
print(f"\nFinal Accuracy: {iter_acc:.2f}%")

## 6. Sparsity 수준별 비교 실험

In [ ]:
print("\n[8] Sparsity 수준별 비교 실험...")
sparsity_levels = [10, 20, 30, 40, 50, 60, 70, 80]

unstructured_results = {}
structured_results = {}
structured_ft_results = {}

for sp in sparsity_levels:
    amount = sp / 100

    # L1 Unstructured
    m = copy.deepcopy(model)
    for _, module in m.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            prune.l1_unstructured(module, name='weight', amount=amount)
    m = remove_pruning(m)
    unstructured_results[sp] = test(m, test_loader, DEVICE)

    # L1 Structured (pruning only - 평가용 복사본에서 mask 제거)
    m = copy.deepcopy(model)
    for _, module in m.named_modules():
        if isinstance(module, nn.Conv2d):
            prune.ln_structured(module, name='weight', amount=amount, n=1, dim=0)
    m_eval = copy.deepcopy(m)
    m_eval = remove_pruning(m_eval)
    structured_results[sp] = test(m_eval, test_loader, DEVICE)

    # L1 Structured + Fine-tune (mask 유지한 채 fine-tuning)
    ft_m = copy.deepcopy(model)
    for _, module in ft_m.named_modules():
        if isinstance(module, nn.Conv2d):
            prune.ln_structured(module, name='weight', amount=amount, n=1, dim=0)
    fine_tune(ft_m, train_loader, FINETUNE_EPOCHS, FINETUNE_LR, DEVICE)
    ft_m = remove_pruning(ft_m)
    structured_ft_results[sp] = test(ft_m, test_loader, DEVICE)

    print(f"    Sparsity {sp}%: Unstruct {unstructured_results[sp]:.2f}% | "
          f"Struct {structured_results[sp]:.2f}% | Struct+FT {structured_ft_results[sp]:.2f}%")

## 7. 시각화

In [ ]:
# 방법별 정확도 비교 바 차트
methods = list(all_accuracies.keys())
accuracies = list(all_accuracies.values())

colors = ['#9E9E9E' if m == 'Baseline' else
          ('#4CAF50' if acc >= baseline_acc - 1 else '#FF9800' if acc >= baseline_acc - 3 else '#F44336')
          for m, acc in zip(methods, accuracies)]

plt.figure(figsize=(14, 6))
bars = plt.bar(methods, accuracies, color=colors, edgecolor='black', linewidth=1.2)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f'{acc:.2f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.xlabel('Method', fontsize=12)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.title('Pruning Methods - Test Accuracy Comparison', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.ylim(min(accuracies) - 5, max(accuracies) + 3)
plt.axhline(y=baseline_acc, color='gray', linestyle=':', alpha=0.5)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Sparsity별 비교
plt.figure(figsize=(12, 6))

sp_keys = list(unstructured_results.keys())
plt.plot(sp_keys, list(unstructured_results.values()), 'bo-', linewidth=2, markersize=8, label='L1 Unstructured')
plt.plot(sp_keys, list(structured_results.values()), 'rs-', linewidth=2, markersize=8, label='L1 Structured')
plt.plot(sp_keys, list(structured_ft_results.values()), 'g^-', linewidth=2, markersize=8, label='L1 Structured + Fine-tune')
plt.axhline(y=baseline_acc, color='gray', linestyle='--', alpha=0.7, label=f'Baseline ({baseline_acc:.2f}%)')

plt.xlabel('Sparsity (%)', fontsize=12)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.title('Pruning Methods: Sparsity vs Accuracy Comparison', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Iterative Pruning 과정 시각화
iterations = [h['iteration'] for h in iter_history]
sparsities = [h['sparsity'] for h in iter_history]
prune_accs = [h['prune_accuracy'] for h in iter_history]
ft_accs = [h['finetune_accuracy'] for h in iter_history]

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(iterations, prune_accs, 'ro--', linewidth=1.5, markersize=8, label='After Pruning')
ax1.plot(iterations, ft_accs, 'go-', linewidth=2, markersize=8, label='After Fine-tuning')
ax1.set_xlabel('Iteration', fontsize=12)
ax1.set_ylabel('Test Accuracy (%)', fontsize=12, color='green')
ax1.tick_params(axis='y', labelcolor='green')
ax1.legend(loc='center left')

ax2 = ax1.twinx()
ax2.bar(iterations, sparsities, alpha=0.2, color='blue', label='Sparsity')
ax2.set_ylabel('Sparsity (%)', fontsize=12, color='blue')
ax2.tick_params(axis='y', labelcolor='blue')
ax2.legend(loc='center right')

plt.title('Iterative Pruning: Accuracy & Sparsity per Iteration', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. 최종 결과 요약

In [ ]:
print("=" * 70)
print("Pruning 실험 결과 요약")
print("=" * 70)
print(f"\n{'Method':<40} {'Accuracy':<12} {'vs Baseline':<12}")
print("-" * 70)

for method, acc in all_accuracies.items():
    diff = acc - baseline_acc if method != 'Baseline' else 0
    diff_str = f"{diff:+.2f}%" if diff != 0 else "-"
    print(f"{method:<40} {acc:>10.2f}% {diff_str:>12}")

print("-" * 70)

non_baseline = {k: v for k, v in all_accuracies.items() if k != 'Baseline'}
if non_baseline:
    best_method = max(non_baseline, key=non_baseline.get)
    best_acc = non_baseline[best_method]
    print(f"\n최고 Pruning 성능: {best_method} ({best_acc:.2f}%)")
    print(f"Baseline 대비 변화: {best_acc - baseline_acc:+.2f}%")

In [ ]:
# 결과 파일 저장
lines = [
    "=" * 70,
    "Pruning 실험 결과 요약",
    "=" * 70, "",
    f"{'Method':<40} {'Accuracy':<12} {'vs Baseline':<12}",
    "-" * 70,
]
for method, acc in all_accuracies.items():
    diff = acc - baseline_acc if method != 'Baseline' else 0
    diff_str = f"{diff:+.2f}%" if diff != 0 else "-"
    lines.append(f"{method:<40} {acc:>10.2f}% {diff_str:>12}")

with open('results/summary.txt', 'w', encoding='utf-8') as f:
    f.write("\n".join(lines))

print("결과 요약 저장: results/summary.txt")

---

## 핵심 인사이트

1. **Unstructured > Structured** (정확도 관점): 개별 가중치 제거가 필터 제거보다 정확도 유지에 유리
2. **Global > Per-layer** (Unstructured): 전역 중요도 기준이 레이어별 기준보다 효과적
3. **Fine-tuning 필수**: Structured pruning 후 fine-tuning으로 상당한 정확도 회복 가능
4. **Iterative > One-shot**: 점진적 pruning이 한 번에 pruning하는 것보다 효과적
5. **Structured의 실용적 장점**: 실제 연산량 감소 및 범용 HW에서의 속도 향상